# TD 3- Premiers modèles avec Scikit-Learn

Ce TD vou montrera comment **utiliser une bibliothèque** pour effectuer un premier apprentissage supervisé (au lieu de faire les algos à la main comme dans les TD précédents).

**Objectifs :**
1. Identifier les variables explicatives `X` et la cible `y`.
2. Construire un jeu d'entraînement et un jeu de test avec `train_test_split()`.
3. Entraîner un modèle avec `fit()` et produire des prédictions avec `predict()`.
4. Évaluer les résultats avec `score()`, l'exactitude (*accuracy*) et une matrice de confusion.
5. Comparer un modèle à une stratégie de référence très simple.

**Matériel :** Python, `pandas`, `scikit-learn`, `matplotlib`. Les données sont fournies directement par Scikit-Learn.
Je vous invite à consulter la documentation en ligne pour comprendre les fonctions recommandées lorsque vous bloquez.

Documentation officielle : 
- [scikit-learn.org](https://scikit-learn.org/stable/)
- https://numpy.org/doc/


> **Mode d'emploi :** exécutez les cellules dans l'ordre, puis complétez les cellules marquées **À vous de jouer**.

## Le parcours d'un modèle

```text
Jeu de données
      ↓
  X (caractéristiques) + y (réponse attendue)
      ↓
Séparation : entraînement / test
      ↓
fit(X_train, y_train)       ← le modèle apprend
      ↓
predict(X_test)            ← le modèle prédit
      ↓
Comparaison à y_test       ← on mesure les erreurs
```

**note :** l'ensemble de test doit rester inconnu du modèle pendant son entraînement. Nous l'utiliserons une fois pour mesurer sa capacité à prédire de nouvelles données.

## 1. Découvrir les données

Notre objectif : **reconnaître une espèce d'iris** à partir de mesures de sa fleur.

C'est un problème de **classification** : la réponse est une catégorie parmi trois espèces, et non un nombre continu (comme dans une régression).

Le jeu de données `iris` est inclus dans Scikit-Learn. Chaque observation correspond à une fleur décrite par quatre mesures (longueur et largeur des sépales et des pétales).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import load_iris

print("Version de Scikit-Learn :", sklearn.__version__)

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.head()

La colonne `target` est la **classe à prédire** : `0`, `1` ou `2` correspondent aux trois espèces. Leurs noms sont fournis par `iris.target_names`. Les autres colonnes sont les **caractéristiques** (*features*) utilisées pour faire la prédiction.

In [ ]:
print("Espèces :", list(iris.target_names))
print("Dimensions du tableau :", df.shape)
print("Nombre de fleurs par espèce :")
print(df["target"].value_counts().sort_index())

### Définir `X` et `y`

Dans Scikit-Learn, on utilise habituellement :
- `X` : le tableau des **caractéristiques** (une ligne par observation, une colonne par caractéristique) ;
- `y` : la série des **réponses attendues** (une valeur par observation).

`X` doit être un tableau à deux dimensions. `y` est généralement un vecteur à une dimension.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

print("Forme de X :", X.shape)
print("Forme de y :", y.shape)
X.head()

### ✍️ À vous de jouer 1 - Comprendre le jeu de données

Dans la cellule suivante :
1. Affichez le nom des colonnes de `X`.
2. Affichez le nombre de fleurs dans `X`.
3. Affichez les trois premières valeurs de `y`.

conseil : utiliser les fonctions columns, head() et tolist()

**Question :** Doit-on conserver `target` dans `X` ? Pourquoi ?

In [ ]:
# Exercice 1 - Complétez ici
# 1. Noms des caractéristiques :

# 2. Nombre d'observations :

# 3. Trois premières étiquettes :

## 2. Séparer entraînement et test

Si nous évaluons le modèle **sur les fleurs qu'il a déjà vues**, nous risquons d'obtenir un score trompeur : il a pu mémoriser des exemples, sans avoir appris une règle généralisable.

On découpe donc nos 150 fleurs en :
- **75 % pour l'entraînement**, afin d'apprendre ;
- **25 % pour le test**, afin d'évaluer le résultat sur des observations inconnues.

`random_state=42` assure une répartition reproductible. `stratify=y` conserve approximativement les proportions des espèces dans chaque ensemble.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Entraînement :", X_train.shape[0], "fleurs")
print("Test :", X_test.shape[0], "fleurs")

### Vérifier le découpage

Les ensembles doivent contenir les mêmes **colonnes**, mais pas les mêmes **lignes**. Dans ce TD, le découpage conserve les indices de pandas, ce qui permet de vérifier que les deux groupes ne se recouvrent pas.

In [ ]:
print("Colonnes identiques :", list(X_train.columns) == list(X_test.columns))
print("Nombre de lignes présentes dans les deux groupes :",
      len(set(X_train.index) & set(X_test.index)))

### ✍️ À vous de jouer 2 - Lire les tailles

À partir des quatre variables `X_train`, `X_test`, `y_train` et `y_test` :
1. Affichez leur propriété `.shape`.
2. Vérifiez que le nombre total d'observations d'entraînement et de test est égal à celui de `X`.

**Question :** pourquoi ne fait-on pas `fit(X_test, y_test)` après l'entraînement ?

In [ ]:
# Exercice 2 - Complétez ici
# Afficher les quatre formes (.shape) :

# Vérifier le nombre total de fleurs :

## 3. Entraîner un modèle

Scikit-Learn propose de nombreux algorithmes, mais ils utilisent une **interface similaire**.

Pour notre première classification, choisissons un **arbre de décision** (`DecisionTreeClassifier`) : il apprend tout seul des règles du type « si la largeur du pétale dépasse telle valeur, alors… ».

Nous limiterons sa profondeur avec `max_depth=3` pour garder un modèle relativement simple. Pas besoin de connaître en détail son fonctionnement interne pour apprendre à utiliser l'interface Scikit-Learn, mais il faudra savoir l'expliquer au moment de la soutenance.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

print("Le modèle est entraîné !")

### Faire des prédictions

`fit()` réalise l'apprentissage. `predict()` applique ensuite le modèle à de nouvelles observations.

In [ ]:
y_pred = model.predict(X_test)
print("10 premières prédictions :", y_pred[:10])
print("10 vraies étiquettes :  ", y_test.to_numpy()[:10])

### ✍️ À vous de jouer 3 - Vérifier quelques prédictions

Construisez un petit tableau pandas avec les **5 premières fleurs de test** et deux colonnes : `Vrai` et `Prédit`. (solutions et résultats du modèle)

Pour extraire les cinq premières valeurs d'un tableau pandas, utilisez `.iloc[:5]`. Pour les prédictions NumPy, utilisez `y_pred[:5]`.

**Question :** Le modèle est-il correct sur ces quelques valeurs ?

In [ ]:
# Exercice 3 - Complétez ici
# Indice : pd.DataFrame({"Vrai": ..., "Prédit": ...})

## 4. Évaluer les résultats

L'**exactitude** (*accuracy*) est la proportion de prédictions correctes :

**Exactitude = nombre de prédictions correctes / nombre total de prédictions**

Pour un **classifieur**, `model.score(X_test, y_test)` renvoie cette exactitude. On peut aussi la calculer directement en comparant `y_pred` à `y_test` avec `accuracy_score()`, c'est un pourcentage de réussite.

**Attention :** pour une régression, la méthode `.score()` n'a pas la même signification (elle renvoie en général le coefficient R²).

In [ ]:
from sklearn.metrics import accuracy_score

score_train = model.score(X_train, y_train)
score_test = model.score(X_test, y_test)
score_pred = accuracy_score(y_test, y_pred)

print(f"Exactitude sur l'entraînement : {score_train:.1%}")
print(f"Exactitude sur le test        : {score_test:.1%}")
print(f"Même score avec accuracy_score : {score_pred:.1%}")

### Comment interpréter les résultats ?

Un bon score sur l'entraînement ne suffit pas : l'objectif est de bien prédire **sur des observations nouvelles**. Il est fréquent d'obtenir un score d'entraînement légèrement supérieur au score de test.

Une autre question se pose : **le modèle fait-il réellement mieux qu'une stratégie "naïve" ? (par exemple répondre toujours la même chose)**

### Comparer à un modèle de référence

`DummyClassifier` peut servir de point de comparaison. Avec `strategy="most_frequent"`, il prédit **toujours la classe la plus fréquente** dans les données d'entraînement, sans tenir compte des mesures des fleurs.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

print(f"Modèle de référence : {baseline.score(X_test, y_test):.1%}")
print(f"Arbre de décision   : {score_test:.1%}")

### Comprendre les erreurs avec une matrice de confusion

La matrice de confusion indique combien de fois chaque **vraie classe** a été prédite comme chaque **classe estimée** :
- la diagonale correspond aux bonnes prédictions ;
- les autres cases correspondent aux erreurs.

Le graphique est produit par Scikit-Learn, et affiché avec `matplotlib` (plt).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=iris.target_names,
    cmap="Blues",
    colorbar=False,
)
plt.title("Matrice de confusion - ensemble de test")
plt.show()

### ✍️ À vous de jouer 4

Répondez **en texte**, sous la cellule :
1. Quelle proportion de prédictions est correcte sur les données de test ?
2. Le modèle de référence est-il meilleur ou moins bon que l'arbre ?
3. D'après la matrice de confusion, quelles espèces sont parfois confondues par le modèle ?
4. Pourquoi un score de **100 % sur les données d'entraînement** ne prouve-t-il pas qu'un modèle est parfait ?

In [ ]:
# Réponses de l'exercice 4 (dans un commentaire ou une cellule Markdown)
# 1.
# 2.
# 3.
# 4.

## 5. Mini-défi autonome

En utilisant **uniquement les données d'entraînement**, entraînez un deuxième arbre avec `max_depth=1`.

1. Créez le modèle.
2. Entraînez-le avec `fit()`.
3. Calculez **son score sur l'entraînement**.
4. Comparez-le au score d'entraînement du premier modèle (`score_train`).

**Question :** pourquoi un arbre plus simple peut-il obtenir un score d'entraînement moins élevé ?

In [ ]:
# Mini-défi - À compléter
# 1. Créer un DecisionTreeClassifier de profondeur 1 :

# 2. Entraîner le modèle :

# 3. Afficher son score sur X_train, y_train :

# 4. Comparer à score_train :

## À retenir

| Outil Scikit-Learn | Son rôle |
|---|---|
| `load_iris()` | Charger un jeu de données d'exemple |
| `train_test_split()` | Répartir les données entre entraînement et test |
| `DecisionTreeClassifier()` | Créer un modèle de classification |
| `model.fit(X_train, y_train)` | Entraîner le modèle |
| `model.predict(X_test)` | Produire des prédictions |
| `model.score(X_test, y_test)` | Évaluer ici l'exactitude du classifieur |
| `accuracy_score(y_test, y_pred)` | Calculer l'exactitude des prédictions |
| `ConfusionMatrixDisplay` | Visualiser les bonnes et mauvaises classifications |

**Trois idées à conserver pour vos projets :**
1. Définissez clairement **ce que vous souhaitez prédire** (`y`) et **à partir de quelles données** (`X`).
2. **Séparez les données avant d'entraîner** un modèle.
3. Un taux de succès élevé n'est pas suffisant : **comparez à une référence**, regardez les erreurs et demandez-vous si le résultat se généralisera.

---
*c'était une version simplifiée du notebook « Introduction to Scikit-Learn » de Daniel Bourke - [source du tutoriel](https://github.com/mrdbourke/zero-to-mastery-ml/blob/master/section-2-data-science-and-ml-tools/introduction-to-scikit-learn.ipynb). Documentation officielle : [scikit-learn.org](https://scikit-learn.org/stable/).*